Mô hình PEAS cho bài toán 8-Puzzle
P (Performance):
Tìm được trạng thái đích với số bước di chuyển ít nhất, giảm thời gian xử lý và hạn chế lặp lại các trạng thái đã đi qua.
E (Environment):
Môi trường gồm bảng 3×3 chứa các ô số từ 1 đến 8 và một ô trống dùng để thực hiện di chuyển.
A (Actuators):
Thực hiện các thao tác di chuyển ô trống theo bốn hướng: trái, phải, lên và xuống.
S (Sensors):
Quan sát trạng thái hiện tại của ma trận, xác định vị trí các ô số, nhận biết trạng thái đích và lưu lại các trạng thái đã duyệt để tránh lặp lại.

Percept (Nhận thức)
Agent quan sát môi trường dưới dạng ma trận 3×3 gồm các số từ 1 đến 8 và một ô trống ký hiệu là 0.
Agent xác định vị trí hiện tại của các ô số và vị trí của ô trống để thực hiện hành động phù hợp.
Ngoài trạng thái hiện tại, agent còn lưu lại các trạng thái đã đi qua để xây dựng mô hình môi trường và tránh lặp lại.
Ví dụ trạng thái đích (Goal State):
1 2 3
4 5 6
7 8 0

Rules (Quy tắc hoạt động)

Giả sử ô trống nằm tại vị trí (x, y):

if x > 0:
    move.append(UP)
(Ô trống không nằm ở hàng trên cùng nên có thể di chuyển lên trên).

if x < 2:
    move.append(DOWN)
(Ô trống không nằm ở hàng dưới cùng nên có thể di chuyển xuống dưới).

if y > 0:
    move.append(LEFT)
(Ô trống không nằm ở cột ngoài cùng bên trái nên có thể di chuyển sang trái).

if y < 2:
    move.append(RIGHT)
(Ô trống không nằm ở cột ngoài cùng bên phải nên có thể di chuyển sang phải).

Agent sẽ lưu các hướng di chuyển hợp lệ vào danh sách move.

Sau đó:
- Kiểm tra trạng thái mới sau mỗi lần di chuyển.
- Nếu trạng thái đã từng xuất hiện trong bộ nhớ thì hạn chế chọn lại.
- Nếu có nhiều hướng hợp lệ, agent có thể chọn ngẫu nhiên (random) một hướng chưa được duyệt để tránh lặp trạng thái.

In [6]:
import time

class ModelBased8Puzzle:

    def __init__(self, initial_state):

        # Ma trận ban đầu
        self.grid = [row[:] for row in initial_state]

        # Trạng thái đích
        self.goal = [
            [1, 2, 3],
            [4, 5, 6],
            [7, 8, 0]
        ]

        # Lưu bước trước đó
        self.last_action = None

    # In ma trận
    def print_grid(self, step):

        print(f"\n===== STEP {step} =====")

        for row in self.grid:
            print(*row)

    # Kiểm tra goal
    def is_goal(self):

        return self.grid == self.goal

    # Tìm ô trống
    def find_blank(self):

        for i in range(3):
            for j in range(3):

                if self.grid[i][j] == 0:
                    return i, j

    # Manhattan Distance
    def distance(self, row, col, value):

        if value == 0:
            return 0

        target_row = (value - 1) // 3
        target_col = (value - 1) % 3

        return abs(row - target_row) + abs(col - target_col)

    # Đánh giá trạng thái
    def evaluate(self, state):

        score = 0

        for i in range(3):
            for j in range(3):

                score += self.distance(i, j, state[i][j])

        return score

    # Hướng ngược lại
    def opposite(self, action):

        opposite_move = {
            "UP": "DOWN",
            "DOWN": "UP",
            "LEFT": "RIGHT",
            "RIGHT": "LEFT"
        }

        return opposite_move[action]

    # Các bước đi hợp lệ
    def possible_moves(self, x, y):

        moves = []

        if x > 0:
            moves.append((-1, 0, "UP"))

        if x < 2:
            moves.append((1, 0, "DOWN"))

        if y > 0:
            moves.append((0, -1, "LEFT"))

        if y < 2:
            moves.append((0, 1, "RIGHT"))

        return moves

    # Model-Based Action
    def model_action(self):

        x, y = self.find_blank()

        moves = self.possible_moves(x, y)

        best_score = float("inf")

        best_grid = None

        best_action = None

        for dx, dy, action in moves:

            # Không quay lại bước vừa đi
            if self.last_action is not None:

                if action == self.opposite(self.last_action):
                    continue

            nx = x + dx
            ny = y + dy

            # Tạo bản sao
            temp = [row[:] for row in self.grid]

            # Hoán đổi vị trí
            temp[x][y], temp[nx][ny] = \
            temp[nx][ny], temp[x][y]

            # Tính điểm
            score = self.evaluate(temp)

            # Chọn trạng thái tốt nhất
            if score < best_score:

                best_score = score

                best_grid = temp

                best_action = action

        # Nếu có trạng thái tốt hơn
        if best_grid is not None:

            self.grid = best_grid

            self.last_action = best_action

            return best_action

        return "NO MOVE"


if __name__ == "__main__":

    # Ma trận cũ
    initial_map = [
        [1, 2, 3],
        [5, 6, 4],
        [8, 7, 0]
    ]

    agent = ModelBased8Puzzle(initial_map)

    step = 1

    # Giới hạn bước để tránh lặp vô hạn
    while not agent.is_goal() and step <= 100:

        agent.print_grid(step)

        action = agent.model_action()

        print("\nAction:", action)

        if action == "NO MOVE":
            break

        step += 1

        time.sleep(0.3)

    print("\n===== FINAL STATE =====")

    for row in agent.grid:
        print(*row)

    if agent.is_goal():

        print("\nGOAL ACHIEVED!")

    else:

        print("\nAgent stopped.")


===== STEP 1 =====
1 2 3
5 6 4
8 7 0

Action: UP

===== STEP 2 =====
1 2 3
5 6 0
8 7 4

Action: LEFT

===== STEP 3 =====
1 2 3
5 0 6
8 7 4

Action: LEFT

===== STEP 4 =====
1 2 3
0 5 6
8 7 4

Action: UP

===== STEP 5 =====
0 2 3
1 5 6
8 7 4

Action: RIGHT

===== STEP 6 =====
2 0 3
1 5 6
8 7 4

Action: DOWN

===== STEP 7 =====
2 5 3
1 0 6
8 7 4

Action: DOWN

===== STEP 8 =====
2 5 3
1 7 6
8 0 4

Action: LEFT

===== STEP 9 =====
2 5 3
1 7 6
0 8 4

Action: UP

===== STEP 10 =====
2 5 3
0 7 6
1 8 4

Action: RIGHT

===== STEP 11 =====
2 5 3
7 0 6
1 8 4

Action: UP

===== STEP 12 =====
2 0 3
7 5 6
1 8 4

Action: LEFT

===== STEP 13 =====
0 2 3
7 5 6
1 8 4

Action: DOWN

===== STEP 14 =====
7 2 3
0 5 6
1 8 4

Action: DOWN

===== STEP 15 =====
7 2 3
1 5 6
0 8 4

Action: RIGHT

===== STEP 16 =====
7 2 3
1 5 6
8 0 4

Action: RIGHT

===== STEP 17 =====
7 2 3
1 5 6
8 4 0

Action: UP

===== STEP 18 =====
7 2 3
1 5 0
8 4 6

Action: UP

===== STEP 19 =====
7 2 0
1 5 3
8 4 6

Action: LEFT

===== STE

KeyboardInterrupt: 